# English → Gujarati Neural Machine Translation (Transformer from scratch) — v2

### What changed from v1 and why

| Problem in v1 | Fix in v2 |
|---|---|
| **Positional encoding bug** — `pe` was indexed by *batch* position (`pe[:x.size(0)]` on batch-first tensors), so every token in a sentence got the same vector and the model had **no word-order information** | Positions are indexed along the sequence dimension |
| Word-level vocab from `.split()` (punctuation glued to words, rich Gujarati morphology → many `<unk>`) | Joint EN+GU **SentencePiece** unigram vocab (32k) with byte fallback — no `<unk>` |
| Only 150k of ~3.07M sentence pairs used → overfit after epoch 8 | Whole corpus, cleaned + deduplicated |
| Fixed padding to 128 tokens, batch of 64 sentences | Length-bucketed **token-based batching** (~25k tokens/update) |
| Constant LR 1e-4, no warmup, no clipping | Inverse-sqrt schedule with 4k warmup, peak 7e-4, grad clipping |
| No label smoothing, post-LN, 4 layers | Label smoothing 0.1, **pre-LN**, 6+6 layers, shared/tied embeddings |
| fp32, plain attention | fp16 mixed precision, fused `scaled_dot_product_attention`, multi-GPU (T4×2) |
| No checkpointing, final (overfit) weights saved | Best checkpoint, **checkpoint averaging**, early stopping, resumable |
| Greedy decoding only; BLEU called with references in the wrong shape | Batched **beam search**; BLEU / chrF++ / TER on a held-out test set |

### How to run on Kaggle
1. **Accelerator:** GPU T4 ×2 (P100 also works, just slower). **Internet:** on (for `pip install`).
2. Add the dataset `english-to-gujarati-machine-translation-dataset` as input (the path is auto-detected).
3. Use **Save Version → Save & Run All (Commit)** so training runs in the background for the full ~10.5h budget.
4. Download **`nmt_en_gu.pt`** from the output and place it next to `app.py`.

**Continuing training in a new session:** add the previous version's output as an input and set `RESUME_CHECKPOINT` to its `last.pt`.

In [ ]:
!pip install -q sacrebleu sentencepiece

In [ ]:
import glob
import itertools
import os
import random
import re
import time
from collections import deque

import numpy as np
import sacrebleu
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(f"Using device: {device} ({NUM_GPUS} GPU(s))")
for i in range(NUM_GPUS):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")
print(f"torch {torch.__version__}, sentencepiece {spm.__version__}, sacrebleu {sacrebleu.__version__}")

## Configuration

In [ ]:
# ---- Paths ----
_found = glob.glob("/kaggle/input/**/train.en", recursive=True)
DATA_DIR = os.path.dirname(_found[0]) if _found else "en-gu"
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("Data dir:", DATA_DIR)

# To continue a previous run: add its output as an input dataset and point this at its last.pt
RESUME_CHECKPOINT = None   # e.g. "/kaggle/input/my-previous-run/last.pt"

SEED = 42
MAX_SENTENCES = None       # None = whole corpus (~3.07M pairs); set e.g. 500_000 for a quick experiment
VAL_SIZE = 2000
TEST_SIZE = 2000
MAX_WORDS = 80             # drop pairs with longer sentences (cleaning)
MAX_TOKENS_PER_SENT = 128  # drop longer training pairs (in subword tokens)

# ---- Tokenizer ----
VOCAB_SIZE = 32000

# ---- Model (Transformer-base) ----
MODEL_CONFIG = dict(
    vocab_size=VOCAB_SIZE,
    d_model=512,
    num_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    d_ff=2048,
    dropout=0.1,
    max_len=256,
)

# ---- Optimisation ----
MAX_TOKENS_PER_GPU = 6144    # padded tokens per GPU per micro-batch; lower if you hit CUDA OOM
TOKENS_PER_UPDATE = 24576    # effective batch size, reached through gradient accumulation
PEAK_LR = 7e-4
WARMUP_STEPS = 4000
LABEL_SMOOTHING = 0.1
CLIP_NORM = 1.0
MAX_EPOCHS = 30
TIME_BUDGET_HOURS = 10.5     # Kaggle stops sessions at 12h; leaves time for averaging, eval and export
EVAL_EVERY = 2000            # updates between validation runs / checkpoints
PATIENCE = 6                 # stop after this many evaluations without val-loss improvement
NUM_SNAPSHOTS_TO_AVERAGE = 5

MAX_TOKENS = MAX_TOKENS_PER_GPU * max(NUM_GPUS, 1)
UPDATE_FREQ = max(1, round(TOKENS_PER_UPDATE / MAX_TOKENS))
print(f"Micro-batch: {MAX_TOKENS} tokens, gradient accumulation: {UPDATE_FREQ} "
      f"-> ~{MAX_TOKENS * UPDATE_FREQ} tokens/update")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Model and decoding code

Written to `nmt_model.py` so the Streamlit app imports **exactly** the same code (the same file sits next to `app.py`).

In [ ]:
%%writefile nmt_model.py
"""Transformer encoder-decoder for English -> Gujarati translation.

This file is shared by the Kaggle training notebook (which writes it with
%%writefile) and the Streamlit app, so training and inference always use the
exact same architecture and decoding code.
"""
import math
import re

import torch
import torch.nn as nn
import torch.nn.functional as F

# These ids are baked into the SentencePiece model trained in the notebook
PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3


def normalize_text(text):
    """Collapse whitespace; SentencePiece handles Unicode normalisation."""
    return re.sub(r"\s+", " ", text).strip()


def sinusoidal_positions(max_len, d_model):
    position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (max_len, d_model): broadcasts over the batch dimension


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.dropout = dropout
        self.w_q = nn.Linear(d_model, d_model)
        self.w_kv = nn.Linear(d_model, 2 * d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, query, memory, mask=None, is_causal=False):
        batch_size, q_len, d_model = query.shape
        k_len = memory.size(1)
        q = self.w_q(query).view(batch_size, q_len, self.num_heads, self.d_k).transpose(1, 2)
        k, v = self.w_kv(memory).view(batch_size, k_len, 2, self.num_heads, self.d_k).permute(2, 0, 3, 1, 4)
        # Fused attention kernel (Flash / memory-efficient) when the GPU supports it
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        return self.w_o(out.transpose(1, 2).reshape(batch_size, q_len, d_model))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))


# Pre-LayerNorm layers: far more stable to train than the original post-LN design
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attention(h, h, src_mask))
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask):
        # Target padding only ever sits after real tokens, so the causal mask is enough
        h = self.norm1(x)
        x = x + self.dropout(self.self_attention(h, h, is_causal=True))
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attention(h, memory, src_mask))
        x = x + self.dropout(self.feed_forward(self.norm3(x)))
        return x


class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, num_encoder_layers=6,
                 num_decoder_layers=6, d_ff=2048, dropout=0.1, max_len=256):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        self.embed_scale = math.sqrt(d_model)

        # One joint EN+GU vocabulary, so a single embedding matrix is shared by the
        # encoder input, decoder input and output projection (fewer params, better generalisation)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.register_buffer("pe", sinusoidal_positions(max_len, d_model), persistent=False)
        self.dropout = nn.Dropout(dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_decoder_layers)
        ])
        self.encoder_norm = nn.LayerNorm(d_model)
        self.decoder_norm = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)

        self._reset_parameters()
        self.output_proj.weight = self.embedding.weight

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=self.d_model ** -0.5)
        with torch.no_grad():
            self.embedding.weight[PAD_IDX].zero_()

    def embed(self, tokens):
        x = self.embedding(tokens) * self.embed_scale + self.pe[: tokens.size(1)]
        return self.dropout(x)

    def encode(self, src):
        src_mask = (src != PAD_IDX)[:, None, None, :]  # (B, 1, 1, S), True = attend
        x = self.embed(src)
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return self.encoder_norm(x), src_mask

    def decode(self, tgt, memory, src_mask):
        x = self.embed(tgt)
        for layer in self.decoder_layers:
            x = layer(x, memory, src_mask)
        return self.output_proj(self.decoder_norm(x))

    def forward(self, src, tgt):
        memory, src_mask = self.encode(src)
        return self.decode(tgt, memory, src_mask)


# ---------------------------------------------------------------------------
# Decoding
# ---------------------------------------------------------------------------

BANNED_TOKENS = [PAD_IDX, UNK_IDX, BOS_IDX]


def _max_new_tokens(model, src, max_len_a, max_len_b):
    src_len = int((src != PAD_IDX).sum(1).max())
    return min(model.max_len - 1, int(max_len_a * src_len + max_len_b))


def _strip(ids):
    if EOS_IDX in ids:
        ids = ids[: ids.index(EOS_IDX)]
    return [i for i in ids if i != PAD_IDX]


@torch.no_grad()
def greedy_decode(model, src, max_len_a=1.5, max_len_b=10):
    max_new = _max_new_tokens(model, src, max_len_a, max_len_b)
    memory, src_mask = model.encode(src)
    ys = torch.full((src.size(0), 1), BOS_IDX, dtype=torch.long, device=src.device)
    finished = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    for _ in range(max_new):
        logits = model.decode(ys, memory, src_mask)[:, -1]
        logits[:, BANNED_TOKENS] = float("-inf")
        next_tok = logits.argmax(-1).masked_fill(finished, PAD_IDX)
        ys = torch.cat([ys, next_tok.unsqueeze(1)], dim=1)
        finished |= next_tok == EOS_IDX
        if finished.all():
            break
    return [_strip(seq) for seq in ys[:, 1:].tolist()]


@torch.no_grad()
def beam_search(model, src, beam_size=5, length_penalty=1.0, max_len_a=1.5, max_len_b=10):
    batch_size, k = src.size(0), beam_size
    device = src.device
    max_new = _max_new_tokens(model, src, max_len_a, max_len_b)

    memory, src_mask = model.encode(src)
    memory = memory.repeat_interleave(k, dim=0)
    src_mask = src_mask.repeat_interleave(k, dim=0)

    seqs = torch.full((batch_size * k, 1), BOS_IDX, dtype=torch.long, device=device)
    scores = torch.full((batch_size, k), float("-inf"), device=device)
    scores[:, 0] = 0.0  # all beams start identical, so only one is alive at the first step
    beam_offset = (torch.arange(batch_size, device=device) * k).unsqueeze(1)
    finished = [[] for _ in range(batch_size)]  # (normalised score, token ids)
    done = [False] * batch_size

    for _ in range(max_new):
        logits = model.decode(seqs, memory, src_mask)[:, -1].float()
        logp = F.log_softmax(logits, dim=-1)
        logp[:, BANNED_TOKENS] = float("-inf")
        vocab = logp.size(-1)

        candidates = (scores.view(-1, 1) + logp).view(batch_size, k * vocab)
        top_scores, top_idx = candidates.topk(2 * k, dim=1)
        top_beam = torch.div(top_idx, vocab, rounding_mode="floor").tolist()
        top_tok = (top_idx % vocab).tolist()
        top_scores = top_scores.tolist()

        next_scores = [[float("-inf")] * k for _ in range(batch_size)]
        next_beam = [[0] * k for _ in range(batch_size)]
        next_tok = [[PAD_IDX] * k for _ in range(batch_size)]
        for b in range(batch_size):
            if done[b]:
                continue
            n = 0
            for score, beam, tok in zip(top_scores[b], top_beam[b], top_tok[b]):
                if score == float("-inf"):
                    break
                if tok == EOS_IDX:
                    hyp = seqs[b * k + beam, 1:].tolist()
                    finished[b].append((score / (len(hyp) + 1) ** length_penalty, hyp))
                else:
                    next_scores[b][n], next_beam[b][n], next_tok[b][n] = score, beam, tok
                    n += 1
                    if n == k:
                        break
            if len(finished[b]) >= k or n == 0:
                done[b] = True
        if all(done):
            break

        select = (beam_offset + torch.tensor(next_beam, device=device)).view(-1)
        seqs = torch.cat([seqs[select], torch.tensor(next_tok, device=device).view(-1, 1)], dim=1)
        scores = torch.tensor(next_scores, device=device)

    results = []
    for b in range(batch_size):
        if not finished[b]:  # hit the length limit: fall back to the alive beams
            for j in range(k):
                score = scores[b, j].item()
                if score > float("-inf"):
                    hyp = seqs[b * k + j, 1:].tolist()
                    finished[b].append((score / max(len(hyp), 1) ** length_penalty, hyp))
        results.append(max(finished[b])[1] if finished[b] else [])
    return results


@torch.no_grad()
def translate(model, sp, sentences, beam_size=5, batch_size=32, max_len_a=1.5, max_len_b=10):
    """Translate a list of English sentences. `sp` is the SentencePiece processor."""
    model.eval()
    device = next(model.parameters()).device
    results = [""] * len(sentences)
    encoded = {}
    for i, sentence in enumerate(sentences):
        sentence = normalize_text(sentence)
        if sentence:
            encoded[i] = sp.encode(sentence)[: model.max_len - 1] + [EOS_IDX]

    order = sorted(encoded, key=lambda i: len(encoded[i]))  # similar lengths -> less padding
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        width = max(len(encoded[i]) for i in idx)
        src = torch.full((len(idx), width), PAD_IDX, dtype=torch.long)
        for row, i in enumerate(idx):
            src[row, : len(encoded[i])] = torch.tensor(encoded[i])
        src = src.to(device)
        if beam_size > 1:
            outputs = beam_search(model, src, beam_size, max_len_a=max_len_a, max_len_b=max_len_b)
        else:
            outputs = greedy_decode(model, src, max_len_a=max_len_a, max_len_b=max_len_b)
        for i, ids in zip(idx, outputs):
            results[i] = sp.decode(ids)
    return results


def load_for_inference(path, device="cpu"):
    """Load the exported checkpoint (config + weights + SentencePiece model in one file)."""
    import sentencepiece as spm

    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    model = Transformer(**ckpt["config"])
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    # The output projection is tied to the embedding, so the export leaves it out
    if unexpected or set(missing) - {"output_proj.weight"}:
        raise RuntimeError(f"Checkpoint mismatch. Missing: {missing}, unexpected: {unexpected}")
    model.to(device).eval()
    sp = spm.SentencePieceProcessor(model_proto=ckpt["spm_model"].numpy().tobytes())
    return model, sp, ckpt.get("metrics", {})

In [ ]:
import importlib
import nmt_model
importlib.reload(nmt_model)
from nmt_model import (PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX, Transformer,
                       normalize_text, translate, load_for_inference)

## Load and clean the data

The corpus is web-mined, so it has misaligned pairs, untranslated lines and duplicates. A cheap filter removes most of them.

In [ ]:
GUJARATI_RE = re.compile(r"[઀-૿]")
LATIN_RE = re.compile(r"[A-Za-z]")


def keep_pair(en, gu):
    if not en or not gu or en == gu:
        return False
    n_en, n_gu = len(en.split()), len(gu.split())
    if n_en > MAX_WORDS or n_gu > MAX_WORDS:
        return False
    if max(n_en, n_gu) > 3 * min(n_en, n_gu) + 2:  # badly misaligned pair
        return False
    if GUJARATI_RE.search(en) or not LATIN_RE.search(en):
        return False
    gu_chars = len(GUJARATI_RE.findall(gu))
    return gu_chars > 0 and len(LATIN_RE.findall(gu)) <= 0.3 * gu_chars  # mostly Gujarati script


def load_data():
    en_lines, gu_lines, seen = [], [], set()
    total = 0
    with open(f"{DATA_DIR}/train.en", encoding="utf-8") as f_en, \
         open(f"{DATA_DIR}/train.gu", encoding="utf-8") as f_gu:
        for en, gu in tqdm(zip(f_en, f_gu), desc="Reading corpus"):
            total += 1
            if MAX_SENTENCES and total > MAX_SENTENCES:
                break
            en, gu = normalize_text(en), normalize_text(gu)
            if not keep_pair(en, gu):
                continue
            key = hash((en, gu))
            if key in seen:
                continue
            seen.add(key)
            en_lines.append(en)
            gu_lines.append(gu)
    print(f"Kept {len(en_lines):,} of {total:,} pairs ({100 * len(en_lines) / total:.1f}%)")
    return en_lines, gu_lines


all_en, all_gu = load_data()

In [ ]:
# Deterministic split (same seed + same data -> same split, which matters when resuming)
perm = np.random.default_rng(SEED).permutation(len(all_en))
test_idx = perm[:TEST_SIZE]
val_idx = perm[TEST_SIZE:TEST_SIZE + VAL_SIZE]
heldout_sources = {all_en[i].lower() for i in np.concatenate([test_idx, val_idx])}
# Keep held-out source sentences out of training so the scores are honest
train_idx = [i for i in perm[TEST_SIZE + VAL_SIZE:] if all_en[i].lower() not in heldout_sources]

train_en = [all_en[i] for i in train_idx]
train_gu = [all_gu[i] for i in train_idx]
val_en = [all_en[i] for i in val_idx]
val_gu = [all_gu[i] for i in val_idx]
test_en = [all_en[i] for i in test_idx]
test_gu = [all_gu[i] for i in test_idx]
del all_en, all_gu, perm

print(f"train: {len(train_en):,}  val: {len(val_en):,}  test: {len(test_en):,}")
for en, gu in zip(train_en[:3], train_gu[:3]):
    print(f"  {en}\n  {gu}\n")

## SentencePiece tokenizer

One joint unigram model for both languages (the scripts don't overlap, but numbers, names and punctuation do). Byte fallback means no token is ever `<unk>`.

In [ ]:
SPM_PREFIX = f"{WORK_DIR}/spm_en_gu"
resume_state = None

if RESUME_CHECKPOINT:
    # Reuse the exact tokenizer the checkpoint was trained with
    resume_state = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=True)
    with open(f"{SPM_PREFIX}.model", "wb") as f:
        f.write(resume_state["spm_model"].numpy().tobytes())
    print(f"Loaded tokenizer from {RESUME_CHECKPOINT}")
else:
    start = time.time()
    spm.SentencePieceTrainer.train(
        sentence_iterator=iter(train_en + train_gu),
        model_prefix=SPM_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type="unigram",
        character_coverage=0.9999,     # drops ultra-rare noise characters; byte fallback covers them
        byte_fallback=True,
        input_sentence_size=2_000_000,
        shuffle_input_sentence=True,
        pad_id=PAD_IDX, unk_id=UNK_IDX, bos_id=BOS_IDX, eos_id=EOS_IDX,
        num_threads=os.cpu_count(),
        minloglevel=2,
    )
    print(f"Trained tokenizer in {(time.time() - start) / 60:.1f} min")

sp = spm.SentencePieceProcessor(model_file=f"{SPM_PREFIX}.model")
assert sp.get_piece_size() == VOCAB_SIZE and sp.pad_id() == PAD_IDX and sp.eos_id() == EOS_IDX

for s in ["I'm feeling very hungry", train_gu[0]]:
    print(s, "->", sp.encode(s, out_type=str))

In [ ]:
def encode_corpus(lines, chunk=200_000):
    """Tokenise into one flat int32 array + offsets (far less RAM than lists of lists)."""
    lengths, pieces = [], []
    for start in tqdm(range(0, len(lines), chunk), desc="Tokenising", leave=False):
        ids = sp.encode(lines[start:start + chunk])
        lengths.extend(len(x) for x in ids)
        pieces.append(np.fromiter(itertools.chain.from_iterable(ids), dtype=np.int32))
    offsets = np.zeros(len(lengths) + 1, dtype=np.int64)
    np.cumsum(lengths, out=offsets[1:])
    return np.concatenate(pieces), offsets


class PairDataset(Dataset):
    def __init__(self, en_lines, gu_lines, max_tokens=None):
        self.src, self.src_off = encode_corpus(en_lines)
        self.tgt, self.tgt_off = encode_corpus(gu_lines)
        src_lens = np.diff(self.src_off)
        tgt_lens = np.diff(self.tgt_off)
        keep = (src_lens > 0) & (tgt_lens > 0)
        if max_tokens:
            keep &= (src_lens + 1 <= max_tokens) & (tgt_lens + 1 <= max_tokens)
            keep &= (tgt_lens <= 3 * src_lens + 5) & (src_lens <= 3 * tgt_lens + 5)
        self.indices = np.nonzero(keep)[0]
        self.src_lens = src_lens[self.indices]
        self.tgt_lens = tgt_lens[self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        return (self.src[self.src_off[j]:self.src_off[j + 1]],
                self.tgt[self.tgt_off[j]:self.tgt_off[j + 1]])


def collate(batch):
    src_width = max(len(s) for s, _ in batch) + 1
    tgt_width = max(len(t) for _, t in batch) + 1
    src = torch.full((len(batch), src_width), PAD_IDX, dtype=torch.long)
    tgt_in = torch.full((len(batch), tgt_width), PAD_IDX, dtype=torch.long)
    tgt_out = torch.full((len(batch), tgt_width), PAD_IDX, dtype=torch.long)
    for row, (s, t) in enumerate(batch):
        s, t = torch.from_numpy(s), torch.from_numpy(t)
        src[row, :len(s)] = s
        src[row, len(s)] = EOS_IDX
        tgt_in[row, 0] = BOS_IDX
        tgt_in[row, 1:len(t) + 1] = t
        tgt_out[row, :len(t)] = t
        tgt_out[row, len(t)] = EOS_IDX
    return src, tgt_in, tgt_out


class TokenBatchSampler(Sampler):
    """Groups sentences of similar length so each batch holds ~max_tokens padded tokens."""

    def __init__(self, src_lens, tgt_lens, max_tokens, shuffle=True, seed=0):
        self.lens = (np.maximum(src_lens, tgt_lens) + 1).astype(np.int64)
        self.max_tokens = max_tokens
        self.shuffle = shuffle
        self.seed = seed
        self.set_epoch(0)

    def set_epoch(self, epoch):
        rng = np.random.default_rng(self.seed + epoch)
        if self.shuffle:  # sort by length, random order among equal lengths
            order = np.lexsort((rng.random(len(self.lens)), self.lens))
        else:
            order = np.argsort(self.lens, kind="stable")
        lens = self.lens[order].tolist()
        batches, batch = [], []
        for idx, length in zip(order.tolist(), lens):  # ascending, so `length` is the batch width
            if batch and length * (len(batch) + 1) > self.max_tokens:
                batches.append(batch)
                batch = []
            batch.append(idx)
        if batch:
            batches.append(batch)
        if self.shuffle:
            rng.shuffle(batches)
        self.batches = batches

    def __iter__(self):
        return iter(self.batches)

    def __len__(self):
        return len(self.batches)


train_data = PairDataset(train_en, train_gu, max_tokens=MAX_TOKENS_PER_SENT)
val_data = PairDataset(val_en, val_gu)
train_sampler = TokenBatchSampler(train_data.src_lens, train_data.tgt_lens, MAX_TOKENS, seed=SEED)
val_sampler = TokenBatchSampler(val_data.src_lens, val_data.tgt_lens, MAX_TOKENS_PER_GPU, shuffle=False)
train_loader = DataLoader(train_data, batch_sampler=train_sampler, collate_fn=collate,
                          num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_data, batch_sampler=val_sampler, collate_fn=collate)

print(f"Training pairs after length filter: {len(train_data):,}")
print(f"Avg tokens  en: {train_data.src_lens.mean():.1f}  gu: {train_data.tgt_lens.mean():.1f}")
print(f"Micro-batches/epoch: {len(train_sampler):,}  ->  updates/epoch: {len(train_sampler) // UPDATE_FREQ:,}")

## Model, optimiser and schedule

In [ ]:
class LossCompute(nn.Module):
    """Computes the loss on each GPU, so DataParallel never gathers the huge logits tensor."""

    def __init__(self, model, label_smoothing):
        super().__init__()
        self.model = model
        self.label_smoothing = label_smoothing

    def forward(self, src, tgt_in, tgt_out):
        with torch.autocast("cuda", dtype=torch.float16, enabled=src.is_cuda):
            logits = self.model(src, tgt_in)
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), tgt_out.view(-1),
                               ignore_index=PAD_IDX, label_smoothing=self.label_smoothing,
                               reduction="sum")
        ntokens = (tgt_out != PAD_IDX).sum()
        return loss.unsqueeze(0), ntokens.unsqueeze(0)


model = Transformer(**MODEL_CONFIG).to(device)
train_module = LossCompute(model, LABEL_SMOOTHING)
if NUM_GPUS > 1:
    train_module = nn.DataParallel(train_module)

optimizer = torch.optim.Adam(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.98), eps=1e-9,
                             fused=device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")


def lr_at(step):
    """Linear warmup, then inverse square-root decay (Vaswani et al.)."""
    step = max(step, 1)
    return PEAK_LR * min(step / WARMUP_STEPS, (WARMUP_STEPS / step) ** 0.5)


step, start_epoch, best_val_loss, history = 0, 0, float("inf"), []
if resume_state is not None:
    model.load_state_dict(resume_state["model_state_dict"])
    optimizer.load_state_dict(resume_state["optimizer_state_dict"])
    scaler.load_state_dict(resume_state["scaler_state_dict"])
    step = resume_state["step"]
    start_epoch = resume_state["epoch"] + 1
    best_val_loss = resume_state["best_val_loss"]
    history = resume_state["history"]
    print(f"Resumed from step {step}, epoch {start_epoch}, best val loss {best_val_loss:.4f}")
    del resume_state

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Evaluation helpers

In [ ]:
@torch.no_grad()
def validation_loss(model):
    """Token-level cross-entropy (no label smoothing) on the validation set."""
    model.eval()
    total, ntokens = 0.0, 0
    for src, tgt_in, tgt_out in val_loader:
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(src, tgt_in)
        total += F.cross_entropy(logits.float().view(-1, logits.size(-1)), tgt_out.view(-1),
                                 ignore_index=PAD_IDX, reduction="sum").item()
        ntokens += (tgt_out != PAD_IDX).sum().item()
    return total / ntokens


def score_translations(model, sources, references, beam_size=1, batch_size=64):
    with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        hypotheses = translate(model, sp, sources, beam_size=beam_size, batch_size=batch_size)
    return {
        "bleu": sacrebleu.corpus_bleu(hypotheses, [references]).score,
        "chrf++": sacrebleu.corpus_chrf(hypotheses, [references], word_order=2).score,
        "ter": sacrebleu.corpus_ter(hypotheses, [references]).score,
        "hypotheses": hypotheses,
    }


def atomic_save(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)


def spm_tensor():
    return torch.frombuffer(bytearray(sp.serialized_model_proto()), dtype=torch.uint8)


def save_training_state(epoch):
    atomic_save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "step": step,
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "history": history,
        "config": MODEL_CONFIG,
        "spm_model": spm_tensor(),
    }, f"{WORK_DIR}/last.pt")

## Training

Stops on whichever comes first: `MAX_EPOCHS`, the time budget, or early stopping. `last.pt` is written at every evaluation, so even an interrupted session leaves a resumable checkpoint.

In [ ]:
QUICK_VAL_SIZE = 500   # greedy BLEU during training on a subset, to keep evaluation fast
train_start = time.time()
deadline = train_start + TIME_BUDGET_HOURS * 3600
snapshots = deque()
bad_evals = 0
stop_reason = None
epoch = start_epoch

print(f"Training for up to {MAX_EPOCHS} epochs / {TIME_BUDGET_HOURS}h ...")
for epoch in range(start_epoch, MAX_EPOCHS):
    train_sampler.set_epoch(epoch)
    train_module.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss, running_count, micro = torch.zeros((), device=device), 0, 0
    epoch_start = time.time()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}", mininterval=10)

    for src, tgt_in, tgt_out in pbar:
        src = src.to(device, non_blocking=True)
        tgt_in = tgt_in.to(device, non_blocking=True)
        tgt_out = tgt_out.to(device, non_blocking=True)

        loss_sum, ntokens = train_module(src, tgt_in, tgt_out)
        loss = loss_sum.sum() / ntokens.sum()
        scaler.scale(loss / UPDATE_FREQ).backward()
        running_loss += loss.detach()
        running_count += 1
        micro += 1
        if micro % UPDATE_FREQ:
            continue

        step += 1
        for group in optimizer.param_groups:
            group["lr"] = lr_at(step)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        if step % 100 == 0:
            pbar.set_postfix(step=step, loss=f"{(running_loss / running_count).item():.3f}",
                             lr=f"{lr_at(step):.2e}")
            running_loss.zero_()
            running_count = 0

        if step % EVAL_EVERY == 0:
            val_loss = validation_loss(model)
            quick = score_translations(model, val_en[:QUICK_VAL_SIZE], val_gu[:QUICK_VAL_SIZE])
            train_module.train()
            hours = (time.time() - train_start) / 3600
            history.append({"step": step, "epoch": epoch, "val_loss": val_loss,
                            "val_bleu": quick["bleu"], "val_chrf": quick["chrf++"], "hours": hours})
            improved = val_loss < best_val_loss
            print(f"\n[step {step:>6} | epoch {epoch + 1} | {hours:.2f}h] val loss {val_loss:.4f} "
                  f"(ppl {np.exp(val_loss):.1f}) | greedy BLEU {quick['bleu']:.2f} "
                  f"chrF++ {quick['chrf++']:.2f}{'  * best' if improved else ''}")
            print(f"    {val_en[0]}\n    -> {quick['hypotheses'][0]}")

            # Rolling weight snapshots for checkpoint averaging
            path = f"{WORK_DIR}/snapshot_{step}.pt"
            torch.save(model.state_dict(), path)
            snapshots.append(path)
            while len(snapshots) > NUM_SNAPSHOTS_TO_AVERAGE:
                os.remove(snapshots.popleft())

            if improved:
                best_val_loss, bad_evals = val_loss, 0
                torch.save(model.state_dict(), f"{WORK_DIR}/best.pt")
            else:
                bad_evals += 1
            save_training_state(epoch)

            if bad_evals >= PATIENCE:
                stop_reason = f"early stopping ({PATIENCE} evaluations without improvement)"
        if time.time() > deadline:
            stop_reason = stop_reason or "time budget reached"
        if stop_reason:
            break

    print(f"Epoch {epoch + 1} finished in {(time.time() - epoch_start) / 60:.1f} min")
    if stop_reason:
        break
else:
    stop_reason = "max epochs reached"

save_training_state(epoch)
print(f"Stopped: {stop_reason}. Total updates: {step:,}")

In [ ]:
import matplotlib.pyplot as plt

if history:
    steps = [h["step"] for h in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(steps, [h["val_loss"] for h in history], marker="o")
    ax1.set(title="Validation loss", xlabel="update")
    ax2.plot(steps, [h["val_bleu"] for h in history], marker="o", label="BLEU")
    ax2.plot(steps, [h["val_chrf"] for h in history], marker="o", label="chrF++")
    ax2.set(title=f"Greedy scores on {QUICK_VAL_SIZE} val sentences", xlabel="update")
    ax2.legend()
    plt.tight_layout()
    plt.show()

## Checkpoint averaging and model selection

Averaging the weights of the last few checkpoints usually gives a free +0.5–1 BLEU. Each candidate is scored on the full validation set and the best one is kept.

In [ ]:
def average_state_dicts(paths):
    avg = None
    for path in paths:
        state = torch.load(path, map_location="cpu", weights_only=True)
        if avg is None:
            avg = {k: v.float().clone() for k, v in state.items()}
        else:
            for k in avg:
                avg[k] += state[k].float()
    return {k: v / len(paths) for k, v in avg.items()}


candidates = {"last": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
if os.path.exists(f"{WORK_DIR}/best.pt"):
    candidates["best"] = torch.load(f"{WORK_DIR}/best.pt", map_location="cpu", weights_only=True)
if len(snapshots) > 1:
    candidates[f"average of last {len(snapshots)}"] = average_state_dicts(list(snapshots))

results = {}
for name, state in candidates.items():
    model.load_state_dict(state)
    results[name] = score_translations(model, val_en, val_gu, beam_size=1)
    print(f"{name:>22}: BLEU {results[name]['bleu']:.2f}  chrF++ {results[name]['chrf++']:.2f}")

chosen = max(results, key=lambda n: results[n]["chrf++"])
model.load_state_dict(candidates[chosen])
model.eval()
print(f"\nSelected: {chosen}")

## Final evaluation on the held-out test set

In [ ]:
greedy = score_translations(model, test_en, test_gu, beam_size=1)
beam = score_translations(model, test_en, test_gu, beam_size=5, batch_size=32)

print(f"Test set ({len(test_en)} sentences)")
print(f"{'':>8}  {'BLEU':>6}  {'chrF++':>6}  {'TER':>6}")
for name, r in [("greedy", greedy), ("beam=5", beam)]:
    print(f"{name:>8}  {r['bleu']:6.2f}  {r['chrf++']:6.2f}  {r['ter']:6.2f}")

print("\nSample test translations (beam=5):")
for i in range(8):
    print(f"EN : {test_en[i]}\nREF: {test_gu[i]}\nHYP: {beam['hypotheses'][i]}\n")

In [ ]:
test_sentences = [
    "I'm feeling very hungry",
    "Can I get your contact number?",
    "What is your name?",
    "The government has announced new schemes for farmers.",
    "Please close the door when you leave.",
    "India won the cricket match by five wickets.",
]
with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
    outputs = translate(model, sp, test_sentences, beam_size=5)
for en, gu in zip(test_sentences, outputs):
    print(f"Input:       {en}\nTranslation: {gu}\n")

## Export for the app

`nmt_en_gu.pt` is self-contained: model config, fp16 weights and the SentencePiece tokenizer in one file. Download it and put it next to `app.py`.

In [ ]:
EXPORT_PATH = f"{WORK_DIR}/nmt_en_gu.pt"
torch.save({
    "config": MODEL_CONFIG,
    # The output projection is tied to the embedding, so it is not stored twice
    "model_state_dict": {k: v.half() for k, v in model.state_dict().items() if k != "output_proj.weight"},
    "spm_model": spm_tensor(),
    "metrics": {
        "checkpoint": chosen,
        "train_updates": step,
        "test_bleu": round(beam["bleu"], 2),
        "test_chrf": round(beam["chrf++"], 2),
        "test_ter": round(beam["ter"], 2),
        "test_sentences": len(test_en),
    },
}, EXPORT_PATH)
print(f"Saved {EXPORT_PATH} ({os.path.getsize(EXPORT_PATH) / 1e6:.0f} MB)")

# Sanity check: reload exactly the way app.py does
reloaded, reloaded_sp, metrics = load_for_inference(EXPORT_PATH, device)
print(metrics)
print(translate(reloaded, reloaded_sp, ["Can I get your contact number?"], beam_size=5))